# LSTM Autoencoder – Anomalieerkennung

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

for _c in [Path.cwd(), Path.cwd() / "04_notebooks"]:
    if (_c / "shared_setup.py").exists():
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

from shared_setup import load_and_prepare, evaluate, alarm_analysis

train_df, test_df, rul_df, useful_sensors, scaler = load_and_prepare()

# Autoencoder-spezifisch: Training nur auf gesunden Zyklen (erste 30%)
healthy_df = train_df[train_df["cycles"] <= train_df["max_cycle"] * 0.3]

print(f"Trainingseinheiten: {train_df['unit_id'].nunique()}")
print(f"Davon gesunde Zyklen (für Training): {len(healthy_df)}")
print(f"Sensoren: {len(useful_sensors)}")

Trainingseinheiten: 100
Davon gesunde Zyklen (für Training): 6144
Sensoren: 14


In [3]:
import torch
from torch import nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, seq_len):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        hidden = hidden.permute(1, 0, 2).repeat(1, self.seq_len, 1)
        output, _ = self.decoder(hidden)
        return self.output_layer(output)

def create_sequences(df, window_size, features):
    x, labels = [], []
    for uid in df["unit_id"].unique():
        unit = df[df["unit_id"] == uid].sort_values("cycles")
        vals = unit[features].values
        for i in range(len(vals) - window_size + 1):
            x.append(vals[i:i + window_size])
            labels.append((uid, unit["cycles"].values[i + window_size - 1]))
    return np.array(x), np.array(labels)

print(LSTMAutoencoder(input_size=len(useful_sensors), hidden_size=32, seq_len=20))

LSTMAutoencoder(
  (encoder): LSTM(14, 32, batch_first=True)
  (decoder): LSTM(32, 32, batch_first=True)
  (output_layer): Linear(in_features=32, out_features=14, bias=True)
)


In [4]:
from torch.utils.data import DataLoader, TensorDataset

# Parameter aus der Gridsearch
window_size = 20
hidden_size = 32
lr          = 0.001
epochs      = 50
threshold   = 0.3

torch.manual_seed(42)

# Sequenzen aus gesunden Zyklen erstellen
X_sequences, _ = create_sequences(healthy_df, window_size, useful_sensors)
X_tensor = torch.FloatTensor(X_sequences)
dataset  = TensorDataset(X_tensor, X_tensor)
loader   = DataLoader(dataset, batch_size=32, shuffle=True)

# Modell, Loss, Optimizer
model     = LSTMAutoencoder(len(useful_sensors), hidden_size, window_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Training mit Early Stopping
best_loss, patience_counter = float("inf"), 0
for epoch in range(epochs):
    epoch_loss = 0.0
    for x_batch, y_batch in loader:
        output = model(x_batch)
        loss   = criterion(output, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(loader)
    if epoch_loss < best_loss - 1e-4:
        best_loss, patience_counter = epoch_loss, 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f"Early stopping bei Epoche {epoch + 1}")
            break

# Rekonstruktionsfehler auf allen Trainingsdaten berechnen
model.eval()
all_sequences, labels = create_sequences(train_df, window_size, useful_sensors)
all_tensor = torch.FloatTensor(all_sequences)
with torch.no_grad():
    reconstructed = model(all_tensor)
    errors = torch.mean((all_tensor - reconstructed) ** 2, dim=(1, 2))

train_result = pd.DataFrame(labels, columns=["unit_id", "cycle"])
train_result["reconstruction_error"] = errors.numpy()
train_result = train_result.merge(
    train_df[["unit_id", "cycles", "RUL"]],
    left_on=["unit_id", "cycle"], right_on=["unit_id", "cycles"],
)
train_result["anomaly_flag"] = train_result["reconstruction_error"] > threshold

y_true = (train_result["RUL"] <= 30).astype(int)
y_pred = train_result["anomaly_flag"].astype(int)
evaluate(y_true, y_pred, "Trainingsdaten")

first_alarm = (
    train_result[train_result["anomaly_flag"] == 1]
    .groupby("unit_id")["RUL"]
    .max()
)
print("\nErster Alarm tritt auf bei RUL =")
print(first_alarm.describe().round(1))

--- Trainingsdaten ---
Precision: 0.8116
Recall:    0.8587
F1:        0.8345

Erster Alarm tritt auf bei RUL =
count    100.0
mean      32.4
std       12.7
min        9.0
25%       21.0
50%       32.5
75%       42.0
max       60.0
Name: RUL, dtype: float64


## Testphase: 100 unbekannte Triebwerke

- Triebwerke **MIT Alarm** sollten niedrige RUL haben
- Triebwerke **OHNE Alarm** sollten hohe RUL haben

In [5]:
model.eval()
all_sequences, labels = create_sequences(test_df, window_size, useful_sensors)
all_tensor = torch.FloatTensor(all_sequences)

with torch.no_grad():
    reconstructed = model(all_tensor)
    errors = torch.mean((all_tensor - reconstructed) ** 2, dim=(1, 2))

test_result = pd.DataFrame(labels, columns=["unit_id", "cycle"])
test_result["reconstruction_error"] = errors.numpy()
test_result["anomaly_flag"] = test_result["reconstruction_error"] > threshold
test_result = test_result.merge(rul_df, on="unit_id", how="left")
alarm_analysis(test_result, rul_df)

Units mit Alarm: 22 / 100

RUL-Verteilung (MIT Alarm):
count    22.0
mean     20.5
std      16.7
min       7.0
25%      10.0
50%      17.5
75%      24.8
max      85.0
Name: RUL, dtype: float64

RUL-Verteilung (OHNE Alarm):
count     78.0
mean      91.1
std       32.5
min       18.0
25%       66.8
50%       95.5
75%      114.8
max      145.0
Name: RUL, dtype: float64


## Hyperparameter-Suche (Konzept)

Optimiert nach **F1-Score** auf Trainingsdaten (RUL <= 30 = "kritisch"):

In [ ]:
param_grid = {
    "window_size": [20, 30],
    "hidden_size": [16, 32],
    "lr":          [0.001, 0.0005],
    "epochs":      [30, 50],
    "threshold":   [0.2, 0.3, 0.4],
}
# 2 x 2 x 2 x 2 x 3 = 48 Kombinationen
# Vollstaendiger Lauf: autoencoder_gridsearch.py
print(param_grid)